Created Schema for medallion architecture

create schema bronze

create schema silver

create schema gold

Loading KARDS , SPAWNABLES and FORECAST data into bronze

In [8]:
table_names = ["KARDS","SPAWNABLES","FORECAST"]


for table in table_names:
    file_string = f"{table}.parquet"
    df = spark.read.parquet(f"Files/{file_string}")
    df.write.format("delta").mode("overwrite").saveAsTable(f"bronze.{table}")

StatementMeta(, 6d127468-4257-4ae9-974b-af56e2d0a75e, 10, Finished, Available, Finished, False)

Transfer bronze raw data to silver layer

create table silver.kards as select * from bronze.kards

create table silver.spawnables as select * from bronze.spawnables

create table silver.forecast as select * from bronze.forecast

Transformation logic from bronze to silver

In [6]:
df = spark.table("bronze.kards")

df_silver = df.select(
    "CardId", "CardName", "CardType", "CardNation", "CardRarity",
    "CardSubType", "CostToPlay", "CostToOperate", "Attack", "HitPoint",
    "Keywords", "CardEffect", "IsVeteran", "VeteranCostToOperate",
    "VeteranAttack", "VeteranHitPoint", "VeteranKeywords", "VeteranEffect",
    "Status", "Expansion", "IsPermanentPool", "IsSpawnable", "IsForecastable"
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.kards")


df = spark.table("bronze.spawnables")

df_silver = df.drop(
    "SpawnImagePaths", "Spawn6KImagePaths", "Spawn9KImagePaths", "Spawn12KImagePaths"
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.spawnables")

df = spark.table("bronze.forecast")

df_silver = df.drop(
    "ForecastCardImagePath", "ParentCardImagePath"
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("silver.forecast")

StatementMeta(, acd7e2b9-9323-497b-b970-49a34f1fd6e4, 8, Finished, Available, Finished, False)

Null check on kards bronze as it is the main table

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum, when

df = spark.table("bronze.KARDS")
null_counts = df.select([spark_sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
for c in df.columns])
display(null_counts) #nulls are legitimate and not 

In [1]:
from pyspark.sql.functions import col

kards = spark.table("silver.kards").filter(col("IsSpawnable") == True) # spark.table(tab).filter(col(column) (some condition))  --filter returns bool value
spawnables = spark.table("silver.spawnables")

spawn_chain = kards.join(spawnables, kards.CardId == spawnables.CardId, "inner") #table1.join(table2 , join_condition , join_type)
spawn_chain = spawn_chain.drop(spawnables.CardId)
spawn_chain.write.format("delta").mode("overwrite").saveAsTable("gold.spawn_chain") #table.format("delta").mode("overwrite").saveAsTable(table_01)


eligible_for_forecast = spark.table("silver.kards").filter(col("IsForecastable") == True)

eligible_for_forecast.write.format("delta").mode("overwrite").saveAsTable("gold.eligible_for_forecast")

veteran_cards = spark.table("silver.kards").filter(col("IsVeteran") == True)

veteran_cards.write.format("delta").mode("overwrite").saveAsTable("gold.veteran_cards")

permanent_pool_cards = spark.table("silver.kards").filter(col("IsPermanentPool") == True)

permanent_pool_cards.write.format("delta").mode("overwrite").saveAsTable("gold.permanent_pool_cards")

tables = ["kards", "spawnables", "forecast"]

for table in tables:
    df = spark.table(f"silver.{table}")
    df.write.format("delta").mode("overwrite").saveAsTable(f"gold.{table}")

StatementMeta(, 5c42fc2a-d28b-4a85-86e2-a27079e8d1be, 3, Submitted, Running, Running, True)